In [299]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [300]:
name = '6_results_exp_ECG/test017_2s_cambio_elec_5CH.csv'
num_CH = 5
CH_first = 0

In [301]:
df = pd.read_csv(name, sep=';', dtype=str)
df["tiempos"]  = df.index
df = df.reset_index(drop=True)
df.columns

Index(['Time', 'SPI (3,2,1,4)', 'tiempos'], dtype='object')

In [302]:
print(df.head(5))

   Time SPI (3,2,1,4)      tiempos
0  0x2f           NaN  0,000016765
1  0x41           NaN  0,000022380
2  0x33           NaN  0,000028625
3  0x00           NaN  0,000080550
4  0x00           NaN  0,000086175


In [303]:
df = df.drop(columns = 'SPI (3,2,1,4)')

In [304]:
df.columns = ['MOSI', 'tiempos']

print(df.head(21))


    MOSI      tiempos
0   0x2f  0,000016765
1   0x41  0,000022380
2   0x33  0,000028625
3   0x00  0,000080550
4   0x00  0,000086175
5   0x34  0,000092440
6   0x28  0,000145305
7   0x64  0,000150935
8   0x30  0,000157215
9   0xd2  0,000208790
10  0x8b  0,000214410
11  0x31  0,000220655
12  0xda  0,000272830
13  0x18  0,000278460
14  0x32  0,000284720
15  0x2f  0,000336765
16  0x21  0,000342390
17  0x33  0,000348655
18  0x00  0,000400910
19  0x00  0,000406525
20  0x34  0,000412770


In [305]:
df['MOSI'] = df['MOSI'].apply(lambda x: int(x, 16))

In [306]:
# Verifica cómo quedó
print(df.head(10))


   MOSI      tiempos
0    47  0,000016765
1    65  0,000022380
2    51  0,000028625
3     0  0,000080550
4     0  0,000086175
5    52  0,000092440
6    40  0,000145305
7   100  0,000150935
8    48  0,000157215
9   210  0,000208790


In [307]:
# Crea un diccionario para mapear los valores
mapeo = {
    48+CH_first: CH_first
}

# Comienza después de la última clave
ultima_clave = max(mapeo.keys())
ultimo_valor = max(mapeo.values())

for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    nuevo_valor = ultimo_valor + i
    mapeo[nueva_clave] = nuevo_valor

print(mapeo)

{48: 0, 49: 1, 50: 2, 51: 3, 52: 4}


In [308]:
# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['MOSI'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(40))


    MOSI      tiempos type
0     47  0,000016765    S
1     65  0,000022380    S
2     51  0,000028625  3.0
3      0  0,000080550    S
4      0  0,000086175    S
5     52  0,000092440  4.0
6     40  0,000145305    S
7    100  0,000150935    S
8     48  0,000157215  0.0
9    210  0,000208790    S
10   139  0,000214410    S
11    49  0,000220655  1.0
12   218  0,000272830    S
13    24  0,000278460    S
14    50  0,000284720  2.0
15    47  0,000336765    S
16    33  0,000342390    S
17    51  0,000348655  3.0
18     0  0,000400910    S
19     0  0,000406525    S
20    52  0,000412770  4.0
21    39  0,000465230    S
22    14  0,000470850    S
23    48  0,000477120  0.0
24   208  0,000529015    S
25   227  0,000534665    S
26    49  0,000540935  1.0
27   218  0,000593045    S
28    33  0,000598670    S
29    50  0,000604925  2.0
30    47  0,000657095    S
31   110  0,000662705    S
32    51  0,000668965  3.0
33     0  0,000721015    S
34     0  0,000726640    S
35    52  0,000732900  4.0
3

In [309]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [310]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


3.0
S
S
4.0


In [311]:
# Diccionario para almacenar los arrays
resultados = {CH_first: []}
resultados_tiempos = {CH_first: []}



# Obtener el valor máximo actual de clave
ultima_clave = max(resultados.keys())

# Agregar N nuevas claves a ambos diccionarios
for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    resultados[nueva_clave] = []
    resultados_tiempos[nueva_clave] = []

# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['MOSI']
        sub_df_times = df.iloc[i+1:i+3]['tiempos']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [312]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [313]:
for i in range (num_CH):
    print(len(resultados[i]))


10752
10780
10776
10808
10786


In [314]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [315]:
arr0 = resultados[0].flatten()

new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []
for i in range(0, len(arr0)-1, 2):
    combined = arr0[i] * 256 + arr0[i+1]
    new_arr0.append(combined)
new_arr0 = np.array(new_arr0)



In [316]:
arr1 = resultados[1].flatten()
for i in range(0, len(arr1)-1, 2):
    combined = arr1[i] * 256 + arr1[i+1]
    new_arr1.append(combined)
new_arr1 = np.array(new_arr1)


In [317]:

arr2 = resultados[2].flatten()
for i in range(0, len(arr2)-1, 2):
    combined = arr2[i] * 256 + arr2[i+1]
    new_arr2.append(combined)
new_arr2 = np.array(new_arr2)

In [318]:

arr3 = resultados[3].flatten()
for i in range(0, len(arr3)-1, 2):
    combined = arr3[i] * 256 + arr3[i+1]
    new_arr3.append(combined)
new_arr3 = np.array(new_arr3)

In [319]:

arr4 = resultados[4].flatten()
for i in range(0, len(arr4)-1, 2):
    combined = arr4[i] * 256 + arr4[i+1]
    new_arr4.append(combined)
new_arr4 = np.array(new_arr4)

In [320]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()


In [321]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [322]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [323]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [324]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [325]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [326]:
ceros = np.zeros(100)  
# FFT
X = np.fft.fft(np.concatenate((ceros,(new_arr-32768))))

# Número de muestras
N = len(X)

# Frecuencias asociadas (eje x)
freqs = np.fft.fftfreq(N, 1/f)

# Magnitud (módulo)
magnitud = np.abs(X)

# Para mostrar solo la mitad positiva (frecuencias positivas)
idxs = freqs >= 0

plt.plot(freqs[idxs], magnitud[idxs])
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de la señal")
plt.show()

NameError: name 'new_arr' is not defined

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = freqs[idxs],
    y=magnitud[idxs],
    mode='lines',
    name='Valores concatenados'
))

fig.update_layout(
    title='FFT',
    xaxis_title='Frecuencia',
    yaxis_title='Magnitud',
    hovermode='x unified'
)

fig.show()